In [ ]:
%pip install matplotlib mne numpy pandas scikit_learn seaborn sktime numba mlflow optuna python-dotenv

In [ ]:
from os import getenv
from dotenv import load_dotenv
load_dotenv()

import mlflow

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sktime.classification.kernel_based import RocketClassifier

from core.converter import SktimeConverter
from core.model_hybrid import EEGHybridExtractor
from core.train import start_training

In [4]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("EEG_Classification")

E:\develope\python\eegia\venv\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///E:/develope/python/eegia/mlruns/961817029434495920', creation_time=1773489193483, experiment_id='961817029434495920', last_update_time=1773489193483, lifecycle_stage='active', name='EEG_Classification', tags={}, workspace='default'>

## Пайплайны моделей

In [5]:
models = {
    "Hybrid": Pipeline([
        ('extractor', EEGHybridExtractor(sfreq=int(getenv("SFREQ")))),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    "Rocket": Pipeline([
        ('convertor', SktimeConverter()),
        ('rocket_classifier', RocketClassifier(random_state=42))
    ]),
    "CSP_LDA": Pipeline([
        ('csp', CSP(log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis())
    ]),
    "CSP_Forest": Pipeline([
        ('csp', CSP(log=True, norm_trace=False)),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
    ])
}

In [6]:
optuna_settings = {
    "Hybrid": {
        'extractor__num_kernels': ("int", 50, 400),
        'classifier__n_estimators': ("int", 100, 500),
        'classifier__max_depth': ("int", 5, 30)
    },
    "Rocket": {
        'rocket_classifier__num_kernels': ("int", 500, 5000)
    },
    "CSP_LDA": {
        'csp__n_components': ("int", 4, 12),
        'csp__reg': ("categorical", ['ledoit_wolf', 'oas', None])
    },
    "CSP_Forest": {
        'csp__n_components': ("int", 4, 12),
        'csp__reg': ("categorical", ['ledoit_wolf', 'oas', None]),
        'classifier__n_estimators': ("int", 100, 500),
        'classifier__max_depth': ("int", 5, 30)
    }
}

# ЗАПУСК ОБУЧЕНИЯ

In [ ]:
res = start_training([1, 2, 3], [], StratifiedKFold(n_splits=5, shuffle=True, random_state=42), models, optuna_settings)


--- Обработка Датасета 1 ---
Загрузка EEG BCI
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R07.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S001\S001R11.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
Extracting EDF parameters from C:\Users\nymphernus\mne_data\MNE-eegbci-data\files\eegmmidb\1.0.0\S002\S002R03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19679  =      0.000 ...   122.994 secs...
E

[I 2026-04-17 11:50:40,068] A new study created in memory with name: no-name-97b9f1ac-4d93-48a2-a2ed-06a10daf977b


Всего эпох после фильтрации: 112
Классы: {np.int64(1): np.int64(54), np.int64(2): np.int64(33), np.int64(3): np.int64(25)}
Оптимизация Optuna: Hybrid
